# AutoGrasp V2 — object-agnostic collection

Collects delicate, consistent grasps on **any object** (not hardcoded to the red block).
Design + rationale: `docs/V2_AUTOGRASP_PLAN.md`. V1 is frozen at tag `v1-pipeline`.

**Run order:** Config → Imports → **OFFLINE SELF-TEST** (no robot) → [robot cells].
The offline self-test proves the whole decision pipeline before any hardware moves.


## 1 · Config — set the object here (no code is object-specific)


In [ ]:
# ── the ONLY things you change per object ──────────────────────────────────
OBJECT_NAME   = "strawberry"        # SAM3 concept prompt + GraspMemory key
TASK          = f"pick up the {OBJECT_NAME}"   # BridgeData-style language label
PART_PROMPT   = None                # e.g. "handle" for part-directed; None = whole object
DELICATE      = True                # cap grip force + 1-retry (produce); False for rigid
INSTANCE_ID   = f"{OBJECT_NAME}_1"  # bump when you swap in a fresh instance (E2 degradation)

# ── hardware / physics constants (rarely change) ──────────────────────────
GRIPPER_MAX_MM   = 105.0
DELICATE_CAP_N   = 12.0             # max seed force on delicate objects
FRICTION_COEF    = 0.7             # fingertip-on-object mu for anti-slip force
TARGET_EPISODES  = 100

# ── coverage: WIDER + randomized than V1's ±6cm (the measured OOD wall) ────
COVERAGE_HALF_M  = 0.09            # ±9cm placements (V1 trained only ±6)
RANDOMIZE_PLACE  = True            # jittered, not grid — diversity over coverage (DROID)

print(f"object={OBJECT_NAME!r}  task={TASK!r}  delicate={DELICATE}  part={PART_PROMPT}")
print(f"coverage ±{COVERAGE_HALF_M*100:.0f}cm  target {TARGET_EPISODES} episodes")


## 2 · Imports (V2 modules — all unit-tested)


In [ ]:
import sys, os, numpy as np, cv2
sys.path.insert(0, os.path.expanduser("~/magpie_control/src"))
sys.path.insert(0, os.path.expanduser("~/magpie_control/scripts"))
from magpie_control.v2.grasp_planner    import plan, priors_from_memory
from magpie_control.v2.state_builder     import build_state, fz_is_live, WrenchCapture, STATE_NAMES
from magpie_control.v2.deformation_check import assess as deform_assess
from magpie_control.v2.weight_estimate   import assess as weigh, reconcile_force
from magpie_control.v2.episode_meta      import EpisodeMeta, PHASES
print("V2 modules loaded. state layout:", STATE_NAMES)


## 3 · OFFLINE SELF-TEST — run this with NO robot

Exercises plan → weigh → state/fz → deformation gate → episode-meta on a gallery of
object shapes. **Must print `READY` before you touch hardware.** Same code as
`scripts/v2_selftest.py`.


In [ ]:
import importlib.util
_spec = importlib.util.spec_from_file_location("v2selftest", os.path.expanduser("~/magpie_control/scripts/v2_selftest.py"))
_st = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_st)
assert _st.run(), "offline pipeline NOT ready — fix before running the robot"
print("\n^ all green — the decision pipeline is sound. Robot cells below.")


## 4 · Robot bringup  ⚠️ hardware

Start the driver stack (verifies the gripper actually publishes state — the 2026-07-21 lesson).


In [ ]:
# In a terminal first:  bash ~/magpie_control/scripts/bringup_gripper_ft.sh
# Then build the node (reuses the V1 collect node class — drivers, cameras, services).
# The V2 additions are the WrenchCapture (real wrist_fz) + GraspMemory priors.
import rclpy
from geometry_msgs.msg import WrenchStamped
# node = <construct the V1 collect Node exactly as v1_collect.ipynb cell c04 does>
# node.ft = WrenchCapture(node, WrenchStamped)      # <-- fixes the all-zero wrist_fz
# assert node.ft.has_reading or node.spin(10) or node.ft.has_reading, "FT sensor silent"
from grasp_memory import GraspMemory
gm = GraspMemory(os.path.expanduser("~/magpie_control/data/grasp_log"))
print("bring up drivers in a terminal, construct `node`, attach node.ft, then continue")


## 5 · Perceive → plan (one object)  ⚠️ hardware

SAM3 concept prompt (object **or** part) → mask → the shape-gated planner. Object-agnostic.


In [ ]:
def perceive_and_plan(node, gm):
    prompt = PART_PROMPT or OBJECT_NAME
    mask   = sam3_mask(node.color, prompt)           # <- existing SAM3 bridge, concept prompt
    depth  = node.depth
    mm_per_px = depth_scale_at_mask(depth, mask)     # <- existing depth→scale helper
    f_prior, w_prior = priors_from_memory(gm, OBJECT_NAME)
    p = plan(mask, mm_per_px, object_name=OBJECT_NAME,
             force_prior_n=f_prior, width_prior_mm=w_prior,
             max_width_mm=GRIPPER_MAX_MM, delicate=DELICATE)
    print(f"[{p.method}] yaw={p.grasp_yaw_deg:.0f}  center_px={tuple(round(v) for v in p.center_px)}  "
          f"width={p.width_mm}mm  seed_force={p.seed_force_n}N  prior={p.have_prior}")
    return p, mask, mm_per_px
# p, mask, mm_per_px = perceive_and_plan(node, gm)


## 6 · Collection loop  ⚠️ hardware

Phase-labeled, wrist_fz recorded, weight-aware force, deformation-gated, guarded place.
Every V2 upgrade wired in; the robot-motion calls reuse the V1 helpers (`node.move`, DeliGrasp, etc.).


In [ ]:
def collect_one_episode(node, gm, target_xy):
    meta   = EpisodeMeta(task=TASK, object_name=OBJECT_NAME, instance_id=INSTANCE_ID)
    frames = []                              # (state, image, action, phase) go to LeRobot recorder

    def rec(phase, action):                  # call each control tick
        meta.set_phase(phase)
        st = build_state(node.tcp_vec(), node.gs.position, node.gs.force, node.ft.fz)
        frames.append((st, node.color.copy(), action, meta.frame()))

    # 1) plan
    meta.set_phase("approach")
    p, mask, mm_per_px = perceive_and_plan(node, gm)

    # 2) approach + center on the GRASP POINT (not the mask centroid — V1 bug)
    meta.set_phase("center")
    grasp_xyz = pixel_to_world(node, p.center_px, mask)          # existing back-projection
    move_above(node, grasp_xyz, yaw=p.grasp_yaw_deg, rec=rec)

    fz_baseline = node.ft.fz                 # empty-gripper baseline for weighing

    # 3) descend + squeeze (DeliGrasp, seeded by prior; delicate cap)
    meta.set_phase("descend"); descend_to(node, grasp_xyz, rec=rec)   # V1-speed near produce
    meta.set_phase("squeeze")
    deligrasp_close(node, prepos_mm=p.prepos_width_mm, seed_force=p.seed_force_n,
                    delicate=DELICATE, rec=rec)                 # DeliGrasp refines force live

    # 4) lift + WEIGH + anti-slip reconcile
    meta.set_phase("lift"); incremental_lift(node, rec=rec)     # re-grip cadence (AX-12 fix)
    w = weigh(fz_baseline, node.ft.fz, delicate_force_cap_n=DELICATE_CAP_N)
    force, safe = reconcile_force(node.gs.force, w, DELICATE_CAP_N)
    if not safe:
        print(f"  [weight] {w.note} -> clamped {force}N; may slip (logged, not crushed)")
    seated_ap = node.gs.position

    # 5) verify (per-object band from the plan) + deformation gate
    held = (p.aperture_band_mm[0] <= seated_ap <= p.aperture_band_mm[1])
    grade = gemini_quality(node.color, OBJECT_NAME) if held else 0.0
    expected_w = p.expected_width_mm if p.have_prior else 0.0    # cold-start = force-only
    deform = deform_assess(expected_w, seated_ap,
                           applied_force_n=node.gs.force, target_force_n=p.seed_force_n)
    keep = bool(held and grade >= 0.6 and deform.ok)
    print(f"  held={held} grade={grade:.2f} gentle={deform.gentleness} ({deform.reason}) "
          f"mass={w.mass_g}g -> KEEP={keep}")

    # 6) guarded gentle PLACE-DOWN — recorded (pick AND place data from day 1)
    meta.set_phase("transport"); carry_to(node, target_xy, rec=rec)
    meta.set_phase("place");     guarded_place(node, fz_contact_n=1.0, rec=rec)   # stop on wrist_fz
    meta.set_phase("release");   open_and_retreat(node, rec=rec)

    # 7) update priors + gate the episode
    if keep:
        gm.update(OBJECT_NAME, true_force=node.gs.force)         # Kalman force prior
        assert fz_is_live(np.stack([f[0] for f in frames])), "wrist_fz FLAT — FT dead, do not keep!"
        save_lerobot_episode(frames, meta.finish(success=True), grade=grade,
                             deform=deform.gentleness, mass_g=w.mass_g)
    else:
        meta.finish(success=False)
    return keep, meta

print("loop defined — drive it over randomized placements toward TARGET_EPISODES")


## Notes
- **Object-agnostic:** only cell 1 is object-specific. Swap `OBJECT_NAME` (and optionally
  `PART_PROMPT`) to collect a different object — the ladder auto-routes by shape.
- **Weight:** measured from wrist_fz each lift; anti-slip force reconciled against the delicate
  cap; heavy+delicate conflicts are flagged, not crushed.
- **Robot helpers** (`sam3_mask`, `deligrasp_close`, `incremental_lift`, `guarded_place`, …) are
  the existing V1 functions — port them from `v1_collect.ipynb` unchanged. This notebook only
  adds the V2 decision layer around them.
- **fz_is_live guard** refuses to save an episode whose wrist_fz is flat — the V1 all-zero bug
  cannot silently recur.
